In [ ]:
import csv
import random
from faker import Faker

# Inicializamos Faker con múltiples configuraciones regionales
# La de España (es_ES) la dejamos como principal para los apellidos y el DNI
fake_es = Faker("es_ES")

# Creamos una lista de generadores para los nombres extranjeros
locales_extranjeros = [
    Faker("en_US"),  # Inglés
    Faker("fr_FR"),  # Francés
    Faker("it_IT"),  # Italiano
    Faker("de_DE"),  # Alemán
]


def generar_dni_valido():
    letras = "TRWAGMYFPDXBNJZSQVHLCKE"
    numero = random.randint(10000000, 99999999)
    letra = letras[numero % 23]
    return f"{numero}{letra}"


usuarios = []

for i in range(1, 1001):
    # Decidimos un 70% de nombres españoles y un 30% de extranjeros (puedes cambiar este ratio)
    if random.random() > 0.30:
        nombre = fake_es.first_name()
    else:
        # Elige un Faker extranjero al azar y genera el nombre
        fake_extranjero = random.choice(locales_extranjeros)
        nombre = fake_extranjero.first_name()

    # Mantenemos los apellidos y el DNI con el formato local para simular residentes
    apellido = fake_es.last_name() + " " + fake_es.last_name()
    dni = generar_dni_valido()

    # Email limpio basado en las variables anteriores
    primer_apellido = apellido.split()[0].lower()
    email = f"{nombre.lower()}.{primer_apellido}{i}@example.com"

    # Limpieza de caracteres conflictivos para entornos de desarrollo/DB
    email = (
        email.replace("ñ", "n")
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ç", "c")
        .replace("ü", "u")
        .replace("ä", "a")
        .replace("ö", "o")
        .replace("ß", "ss")
    )

    usuarios.append(
        {
            "id": i,
            "nombre": nombre,
            "apellido": apellido,
            "dni": dni,
            "email": email,
        }
    )

# Guardar en CSV
with open("../data/raw/usuarios_mixtos.csv", mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["id", "nombre", "apellido", "dni", "email"]
    )
    writer.writeheader()
    writer.writerows(usuarios)

print(
    f"¡Hecho! Generados {len(usuarios)} registros con un mix de nombres internacionales."
)

¡Hecho! Generados 1000 registros con un mix de nombres internacionales.


In [ ]:
import pandas as pd

# 1. Asumiendo que ya tienes tu DataFrame original con los 1000 usuarios
# df_usuarios = ... (el dataframe generado con Faker)

# 2. Cargamos el nuevo CSV de fraude y riesgo país
ruta_fraude = "../data/raw/dataset_fraude_riesgo_pais.csv"
df_usuarios = pd.read_csv("../data/raw/usuarios_mixtos.csv")  # Cargamos el DataFrame de usuarios
df_fraude = pd.read_csv(ruta_fraude)

# 3. Columnas específicas que necesitamos extraer + la clave de unión
columnas_interes = [
    "id_usuario",
    "email_verificado",
    "dias_antiguedad_cuenta",
    "pais_emision",
    "paso_3d_secure",
]

# Filtramos el dataframe de fraude para traer solo las columnas requeridas
df_fraude_filtrado = df_fraude[columnas_interes]

# 4. Hacemos el merge relacionando 'id' (de tu df) con 'id_usuario' (del nuevo csv)
df_final = df_usuarios.merge(
    df_fraude_filtrado, left_on="id", right_on="id_usuario", how="left"
)

# 5. Opcional: Como ya tienes la columna 'id', la columna 'id_usuario' queda duplicada.
# Podemos eliminarla para dejar el DataFrame limpio.
df_final = df_final.drop(columns=["id_usuario"])

# Verificamos el resultado
print("Columnas resultantes:", df_final.columns.tolist())
print(df_final.head())

Columnas resultantes: ['id', 'nombre', 'apellido', 'dni', 'email', 'email_verificado', 'dias_antiguedad_cuenta', 'pais_emision', 'paso_3d_secure']
   id nombre          apellido        dni                      email  \
0   1   Vera  Cortina Chamorro  32090434Y  vera.cortina1@example.com   
1   1   Vera  Cortina Chamorro  32090434Y  vera.cortina1@example.com   
2   1   Vera  Cortina Chamorro  32090434Y  vera.cortina1@example.com   
3   1   Vera  Cortina Chamorro  32090434Y  vera.cortina1@example.com   
4   1   Vera  Cortina Chamorro  32090434Y  vera.cortina1@example.com   

   email_verificado  dias_antiguedad_cuenta pais_emision  paso_3d_secure  
0                 1                     370           RU               0  
1                 1                     370           ES               0  
2                 1                     370           ES               0  
3                 1                     370           FR               0  
4                 1                     370  

In [5]:
df_final

,id,nombre,apellido,dni,email,email_verificado,dias_antiguedad_cuenta,pais_emision,paso_3d_secure
0,1,Vera,Cortina Chamorro,32090434Y,vera.cortina1@example.com,1,370,RU,0
1,1,Vera,Cortina Chamorro,32090434Y,vera.cortina1@example.com,1,370,ES,0
2,1,Vera,Cortina Chamorro,32090434Y,vera.cortina1@example.com,1,370,ES,0
3,1,Vera,Cortina Chamorro,32090434Y,vera.cortina1@example.com,1,370,FR,0
4,1,Vera,Cortina Chamorro,32090434Y,vera.cortina1@example.com,1,370,US,1
...,...,...,...,...,...,...,...,...,...
10058,1000,Henri,Galván Mulet,12377800M,henri.galvan1000@example.com,1,1282,RU,0
10059,1000,Henri,Galván Mulet,12377800M,henri.galvan1000@example.com,1,1282,RU,0
10060,1000,Henri,Galván Mulet,12377800M,henri.galvan1000@example.com,1,1282,RU,1
10061,1000,Henri,Galván Mulet,12377800M,henri.galvan1000@example.com,1,1282,ES,0


In [ ]:
import pandas as pd

# 2. Cargamos los archivos
ruta_fraude = "../data/raw/dataset_fraude_riesgo_pais.csv"
df_usuarios = pd.read_csv("../data/raw/usuarios_mixtos.csv")
df_fraude = pd.read_csv(ruta_fraude)

# 3. Columnas específicas que necesitamos extraer
columnas_interes = [
    "id_usuario",
    "email_verificado",
    "dias_antiguedad_cuenta",
    "pais_emision",
    "paso_3d_secure",
]

# --- NUEVA LÓGICA: Limpieza de duplicados seleccionando al azar ---
# 1. Filtramos las columnas que nos interesan.
# 2. .sample(frac=1, random_state=42) desordena las filas al azar.
# 3. .drop_duplicates(subset=['id_usuario'], keep='first') se queda con la primera fila que encuentra de cada usuario (que ahora es una aleatoria).
df_fraude_unico = (
    df_fraude[columnas_interes]
    .sample(frac=1, random_state=42)
    .drop_duplicates(subset=["id_usuario"], keep="first")
)

# 4. Hacemos el merge con los datos ya limpios (1 fila por usuario)
df_final = df_usuarios.merge(
    df_fraude_unico, left_on="id", right_on="id_usuario", how="left"
)

# 5. Eliminamos la columna repetida
df_final = df_final.drop(columns=["id_usuario"])

# Verificamos el resultado (debería darte exactamente 1000 filas)
print(f"Total de registros finales: {len(df_final)}")
print("Columnas resultantes:", df_final.columns.tolist())
print(df_final.head())

Total de registros finales: 1000
Columnas resultantes: ['id', 'nombre', 'apellido', 'dni', 'email', 'email_verificado', 'dias_antiguedad_cuenta', 'pais_emision', 'paso_3d_secure']
   id     nombre          apellido        dni  \
0   1       Vera  Cortina Chamorro  32090434Y   
1   2  Françoise   Espinosa Azorin  88803830W   
2   3  Victorino    Alcalá Arteaga  98488133V   
3   4  Hortensia   Martin Pellicer  79194582Q   
4   5    Juanito      Sureda Baena  80341994G   

                             email  email_verificado  dias_antiguedad_cuenta  \
0        vera.cortina1@example.com                 1                     370   
1  francoise.espinosa2@example.com                 1                     363   
2    victorino.alcala3@example.com                 1                     591   
3    hortensia.martin4@example.com                 1                     121   
4      juanito.sureda5@example.com                 1                    1045   

  pais_emision  paso_3d_secure  
0          

In [7]:
df_final

,id,nombre,apellido,dni,email,email_verificado,dias_antiguedad_cuenta,pais_emision,paso_3d_secure
0,1,Vera,Cortina Chamorro,32090434Y,vera.cortina1@example.com,1,370,ES,0
1,2,Françoise,Espinosa Azorin,88803830W,francoise.espinosa2@example.com,1,363,ES,0
2,3,Victorino,Alcalá Arteaga,98488133V,victorino.alcala3@example.com,1,591,ES,0
3,4,Hortensia,Martin Pellicer,79194582Q,hortensia.martin4@example.com,1,121,CN,1
4,5,Juanito,Sureda Baena,80341994G,juanito.sureda5@example.com,1,1045,ES,0
...,...,...,...,...,...,...,...,...,...
995,996,Thierry,Feijoo Nieto,80622961A,thierry.feijoo996@example.com,1,502,ES,0
996,997,William,Jiménez Lastra,57417388G,william.jimenez997@example.com,1,800,ES,1
997,998,Marcial,Pineda Diéguez,64933395W,marcial.pineda998@example.com,1,1104,IT,0
998,999,Mary,Ocaña Cáceres,46498781A,mary.ocana999@example.com,1,365,DE,0


In [9]:
df_final.to_csv('../data/processed/base_users.csv', index=False)